## Token Embeddings

Once the input text has been tokenized and transformed into integer token IDs, the next step is to convert these discrete symbols into **dense, high-dimensional vectors**. These vectors are called **token embeddings**, and they form the very first layer of a transformer-based architecture such as GPT.

A language model like GPT cannot operate directly on integer token IDs. Neural networks are designed to process continuous numerical inputs, typically vectors or matrices. Token embeddings provide this continuous representation by mapping each token ID to a vector in a learnable **embedding space**.

Formally, we define a **token embedding matrix**:

$$
\mathbf{E} \in \mathbb{R}^{V \times d}
$$

* $V$ is the vocabulary size (total number of unique tokens).
* $d$ is the embedding dimension (e.g., 768, 1024, 2048 depending on model size).

For each token ID $i \in \{0, \ldots, V-1\}$, the corresponding embedding is the $i$-th row $\mathbf{E}_i \in \mathbb{R}^d$.

> **Understanding the Embedding Dimension ($d$)**
>
> The **embedding dimension**, denoted $d$, is the number of real-valued features used to represent each token in the vocabulary. It determines the **geometry and capacity** of the space in which tokens are embedded and is one of the most consequential hyperparameters in language modeling.
>
> Conceptually, the embedding dimension can be thought of as defining the size of the “semantic space” in which tokens reside. Each token is represented as a vector $\mathbf{e}_i \in \mathbb{R}^d$, and during training, the model adjusts these vectors so that semantically related tokens end up **closer together** under some metric (e.g., usually cosine similarity or Euclidean distance).
>
> ---
>
> **Why not just use 1-dimensional embeddings?**
>
> One-dimensional embeddings (i.e., scalar values) cannot capture the multi-faceted nature of linguistic tokens. A word like “bank” has multiple meanings (riverbank, financial institution), and a single scalar cannot encode this ambiguity. Increasing the dimensionality allows the embedding to capture multiple *latent factors* simultaneously—e.g., syntax, semantics, morphology, and word frequency.
>
> ---
>
> **How is the embedding dimension chosen?**
>
> There is no fixed rule, but the embedding dimension is typically chosen based on model size and training data scale. Larger values of $d$ provide more expressive capacity but:
>
> * Increase the number of trainable parameters: $\text{params} = V \times d$
> * Increase GPU memory and computational cost
> * May lead to overfitting if the data is small
>
> Typical configurations include:
>
> | Model Tier | Embedding Dimension ($d$) | Example Models           |
> | ---------- | ------------------------- | ------------------------ |
> | Small      | 128 – 512                 | GPT-2 small, DistilGPT   |
> | Medium     | 768 – 1024                | BERT base, GPT-2 medium  |
> | Large      | 2048 – 4096               | GPT-3, LLaMA-2, Claude 1 |
> | Very Large | 8192 – 16384              | GPT-4, Gemini, Claude 3  |
>
> ---
>
> **How does it affect learning?**
>
> * Embeddings with **too small** $d$ may lack sufficient capacity to distinguish nuanced token relationships (underfitting).
> * Embeddings with **too large** $d$ may overfit to surface patterns and require strong regularization.
> * **Optimal $d$** balances representational power with generalization.
>
> In practice, embedding layers are often initialized randomly and then trained alongside the rest of the model using backpropagation. The learning algorithm reshapes the embedding matrix so that the token vectors reflect the statistical structure of the training corpus.
>
> ---
>
> **Theoretical View:**
>
> From a manifold learning perspective, the embedding space attempts to project discrete tokens onto a continuous **manifold** embedded in $\mathbb{R}^d$. The geometry of this manifold (e.g., its curvature, local density, and volume) is crucial to the model's ability to interpolate between token meanings and generalize to unseen combinations.



---

### Initialization of Embedding Weights

At the beginning of training, we initialize the embedding weights **randomly**, typically using a uniform or normal distribution. This is essential because:

* It gives the model a *starting point* for learning.
* It avoids symmetry—ensuring that different tokens begin with different representations.
* It allows gradient descent to incrementally shape the space to reflect semantic similarity.

Common initialization strategies include:

* Xavier/Glorot Initialization
* Kaiming (He) Initialization
* Uniform or Normal sampling bounded by $\sqrt{1/d}$

In PyTorch, this is elegantly encapsulated in the `nn.Embedding` module:

---

In [1]:
import torch
import torch.nn as nn

# Assume a vocabulary size of 10 000 and embedding dimension of 512
vocab_size = 10000
embedding_dim = 512

# Create a trainable embedding matrix
token_embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

# Example: token IDs for a batch of 3 sequences of length 4
example_batch = torch.tensor([
    [12, 483, 94, 5],
    [0, 3, 8, 76],
    [50, 91, 21, 4]
], dtype=torch.long)

# Convert token IDs to embeddings
embedded_batch = token_embedding(example_batch)

print("Input shape     :", example_batch.shape)      # (3, 4)
print("Embedding shape :", embedded_batch.shape)     # (3, 4, 512)

Input shape     : torch.Size([3, 4])
Embedding shape : torch.Size([3, 4, 512])


---
Each token in the batch is now mapped to a 512-dimensional vector. The resulting 3D tensor of shape `(batch_size, sequence_length, embedding_dim)` becomes the **input to the transformer layers**.

* `batch_size = 3`: number of sequences in the batch
* `sequence_length = 4`: number of tokens per sequence
* `embedding_dim = 512`: dimensionality of each token vector

This structure ensures compatibility with **multi-head self-attention**, which expects inputs of the shape $(B, T, d)$ — Batch, Tokens, Dimension.

---

Token embeddings are crucial because:

1. They allow the model to learn **semantic similarities** between tokens (e.g., "king" and "queen" should end up close in vector space).
2. They serve as the **interface** between symbolic text and differentiable computations.
3. They allow gradient-based optimization to adjust the representations throughout training.

Without embeddings, the model would be unable to learn meaningful representations of language, as the numeric token IDs alone are **arbitrary** and carry no information about the content or function of a word.

### Hands-On Example: From Token ID to Embedding Vector

Let’s walk through a complete and carefully annotated example using:

* A **vocabulary size $V = 10$** (token IDs from 0 to 9)
* An **embedding dimension $d = 4$**

---

#### Step 1: Define the Embedding Layer

In [3]:
import torch
import torch.nn as nn

# Step 1: Define vocabulary size and embedding dimension
vocab_size = 10
embedding_dim = 4

# Step 2: Create the embedding layer
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Step 3: Print the weight matrix (each row = embedding of a token ID)
print("Embedding weight matrix (before training):")
print(embedding_layer.weight)

Embedding weight matrix (before training):
Parameter containing:
tensor([[-0.4841,  0.5512, -0.4836, -0.9400],
        [-1.1458,  0.2821, -1.0775, -0.4472],
        [-0.8304, -1.4757,  1.9597, -0.5437],
        [-0.6061, -1.9022, -0.6415, -0.6396],
        [-0.8536, -1.4444,  1.2542, -0.6266],
        [-0.1018,  1.1455,  0.5157,  0.3244],
        [ 0.1347,  1.7384, -1.1711,  0.0822],
        [-1.7157,  0.3199, -0.6921, -1.2236],
        [ 0.8748,  0.3450, -0.0324,  0.8851],
        [-0.3244,  2.8406,  0.1605, -1.1643]], requires_grad=True)


---

> **Understanding the Weight Matrix**
>
> The embedding layer maintains a trainable **weight matrix** $\mathbf{E} \in \mathbb{R}^{10 \times 4}$:
>
> * **10 rows** — one for each token in the vocabulary.
> * **4 columns** — each representing a distinct latent feature of the token’s learned representation.
>
> At initialization, this matrix is filled with **small random values**, typically drawn from a uniform or normal distribution. These values will be **refined through backpropagation** during training, becoming increasingly semantically meaningful.
>
> This matrix is equivalent to a lookup table where each row serves as the vector representation of the corresponding token ID.

---

#### Step 2: Token Lookup via Indexing

Now let us pick a specific token ID and retrieve its embedding vector:

In [4]:
# Step 4: Choose a token ID (e.g., ID = 7)
token_id = torch.tensor([7])

# Step 5: Look up the embedding vector using the embedding layer
embedding_vector = embedding_layer(token_id)

# Step 6: Display the result
print(f"\nEmbedding vector for token ID {token_id.item()}:")
print(embedding_vector)


Embedding vector for token ID 7:
tensor([[-1.7157,  0.3199, -0.6921, -1.2236]], grad_fn=<EmbeddingBackward0>)


---

> **Interpretation: What Does This Do?**
>
> This call:
>
> ```python
> embedding_vector = embedding_layer(token_id)
> ```
>
> retrieves the **7th row** (Python uses 0-based indexing) from the weight matrix. The output is a 4-dimensional tensor representing the embedding vector for token ID 7.
>
> This operation is conceptually equivalent to one-hot encoding the token ID, multiplying that by the weight matrix, and returning the result. PyTorch performs this much more efficiently through direct indexing.

---

#### One-Hot Equivalence Demonstration (Optional)

Let’s show explicitly how this works using matrix multiplication and one-hot encoding:

In [5]:
# Construct a one-hot encoding of token ID 7
one_hot = torch.zeros(vocab_size)
one_hot[7] = 1.0
one_hot = one_hot.view(1, -1)  # reshape for matmul

# Multiply by the embedding weight matrix
manual_embedding = one_hot @ embedding_layer.weight

print("\nManual embedding (via one-hot vector × weight matrix):")
print(manual_embedding)


Manual embedding (via one-hot vector × weight matrix):
tensor([[-1.7157,  0.3199, -0.6921, -1.2236]], grad_fn=<MmBackward0>)


---

> **Embedding as a Matrix Product**
>
> This shows that:
>
$$
\text{embedding vector} = \text{one hot}(7) \cdot \mathbf{E}
$$
>
> where the result is identical to what the embedding layer returns directly.
>
> This is why we can say:
>
> > **The embedding layer is a learnable lookup mechanism implemented as efficient one-hot vector multiplication.**
>
> This process is fully differentiable and can be optimized using the backpropagation algorithm (just like every other layer in a neural network).

---


| Concept                    | Description                                          |
| -------------------------- | ---------------------------------------------------- |
| **Token ID**               | Discrete integer, e.g., 7                            |
| **Embedding Matrix Shape** | $10 \times 4$: 10 tokens, each mapped to a 4D vector |
| **Embedding Operation**    | `embedding_vector = embedding_matrix[token_id]`      |
| **Output Shape**           | $(1, 4)$ — a single 4-dimensional vector             |
| **Parameter Type**         | Learnable weights, updated by gradient descent       |
| **Efficiency**             | Faster than one-hot encoding + matrix multiplication |


> You can think of an embedding vector as a **continuous analog** of a one-hot vector. Whereas one-hot encoding assigns a unique axis-aligned vector to each token, embeddings allow for **dense, overlapping** representations that can encode semantic, syntactic, and contextual properties of tokens.
>
> In information-theoretic terms, the embedding process serves to **compress discrete identity** into a **latent space** (akin to dimensionality reduction), but the axes are **not fixed**. Instead, they are learned to reflect the needs of the downstream transformer model.
>
> Thus, the embedding layer enables:
>
> 1. **Semantic generalization**: similar tokens acquire similar vectors.
> 2. **Efficient computation**: enables batched GPU processing.
> 3. **Model scalability**: embeddings scale linearly with vocabulary and dimension.

---

### Next Steps

* **Add position encodings** to preserve sequence order.
* **Pass the embedded input** through the transformer stack for self-attention.
* **Track gradients** with `.backward()` to update embeddings via backpropagation.